In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.font_manager as fm
from pyproj import Transformer

In [ ]:
# 그래프 시각화를 위한 폰트 설정
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
font_list[:10]  # 일부만 출력
font_path='C:\\Windows\\Fonts'
mpl.rcParams['font.family'] = 'Malgun Gothic'

## 분석내용
#### 취약계층 비율
    - 기초생활수급자 비율 ○
    - 차상위계층 비율 ○
    - 장애인 비율 ○
    - 65세 고령자 비율 ○
    - 상수도 보급률 ○
    - 하수도 보급률 ○
#### 주택취약도
    - 30년이상 노후주택 비율 ○
    - 슬레이트 주택비율 ○
    - 전체 호수 대비 공가/폐가 비율 ○
#### 밀집도
    - 사업지역 내 전체주탣호수(밀집도)

# 데이터 전처리

In [ ]:
#결과 데이터프레임 생성
code = pd.read_excel('../P1_Data/01.External/추가데이터/10. 리 단위 전체 주택별 주소정보(건축물대장).xlsx',skiprows=7)
code = code[['시군구코드','법정동코드','법정동코드명','주용도코드명']]
code = code[code['주용도코드명'].isin(['단독주택', '공동주택'])]
code['코드'] = code['시군구코드'].astype(str) + code['법정동코드'].astype(str)
code[['행정기관', '리']] = code['법정동코드명'].str.split(' ', n=1, expand=True)
pre_code = code[['코드','행정기관','리']]
result = pre_code.drop_duplicates(subset=['코드','행정기관','리'])
result = result.dropna(subset=['리'])
result

In [ ]:
# 총인구 전처리 파일(수작업)
pop = pd.read_excel('../P1_Data/02.Pretreatment/people(수작업).xlsx')
total_pop = pop.groupby(['행정기관','리'])['세대수','인구수'].sum().reset_index()
total_pop.to_excel('../P1_Data/02.Pretreatment/pre_people.xlsx',index=False)

In [ ]:
#인구통계(demographics) 전처리 파일(수작업)
demographics = pd.read_excel('../P1_Data/02.Pretreatment/demographics.xlsx')
re_demographics = pd.merge(total_pop,demographics,on=['행정기관','리'])
re_demographics['기초생활수급자_비율'] = ((re_demographics['기초생활보장'] / re_demographics['인구수'])* 100).round(2)
re_demographics['차상위계층_비율'] = ((re_demographics['차상위계층'] / re_demographics['인구수'])* 100).round(2)
re_demographics['장애인_비율'] = ((re_demographics['장애인'] / re_demographics['인구수'])* 100).round(2)
re_demographics['65세이상고령자_비율'] = ((re_demographics['65세이상인구'] / re_demographics['인구수'])* 100).round(2)

In [ ]:
result_demographics = re_demographics[['행정기관','리','기초생활수급자_비율','차상위계층_비율','장애인_비율','65세이상고령자_비율']]

In [ ]:
#결과
result = pd.merge(result,result_demographics,on=['행정기관','리'])
result

In [ ]:
'''
#리단위 전체 주택
house = pd.read_excel('../P1_Data/01.External/추가데이터/10. 리 단위 전체 주택별 주소정보(건축물대장).xlsx',skiprows=7)
pre_house = house[['법정동코드명','도로명주소','주용도코드명']]
pre_house = pre_house[pre_house['주용도코드명'].isin(['단독주택', '공동주택'])]
# 이름을 띄어쓰기로 분리하여 새로운 열 생성
pre_house[['행정기관', '리']] = pre_house['법정동코드명'].str.split(' ', n=1, expand=True)
pre_house = pre_house[['행정기관','리','도로명주소','주용도코드명']]
# 행정기관과 리컬럼을 기준으로 행의 개수(count) 계산
count_house = pre_house.groupby(['행정기관', '리']).size().reset_index(name='전체주택수')
'''

In [ ]:
count_house

In [ ]:
# 빈집 비율
up30_house = pd.read_excel('../P1_Data/01.External/빈집리스트(도시, 농촌).xlsx',skiprows=1)
pre_up30_house = up30_house[['소재지','건축년도']]
splitted = pre_up30_house['소재지'].str.split(' ', expand=True)
pre_up30_house['행정기관'] = splitted[2]
pre_up30_house['리'] = splitted[3]
# "행정기관" 컬럼 값이 "동"으로 끝나는 행 삭제
pre_up30_house2 = pre_up30_house[~pre_up30_house['행정기관'].str.endswith('동')]
null_house = pre_up30_house2.groupby(['행정기관','리']).size().reset_index(name='빈집수')

re_null_house = pd.merge(count_house,null_house,on=['행정기관','리'],how='outer')
re_null_house['빈집수'] = re_null_house['빈집수'].fillna(0.0)
re_null_house['빈집_비율'] = ((re_null_house['빈집수'] / re_null_house['전체주택수'])* 100).round(2)

result_nullhouse = re_null_house[['행정기관','리','빈집_비율']]
#결과
result = pd.merge(result,result_nullhouse,on=['행정기관','리'])
result

In [ ]:
#슬레이트 주택비율
## slate(수작업).xlsx :  건축물 용도가 주택에 해당하고, 슬레이트 철거여부가 미철거에 해당하는 row만 추출
slate = pd.read_excel('../P1_Data/02.Pretreatment/slate(수작업).xlsx')
pre_slate = slate[['주소_지번','건축물           용도','슬레이트            사용부위','철거           여부']]
pre_slate.columns = ['주소','건축물 용도','슬레이트 사용부위','철거여부']
pre_slate = pre_slate.dropna(subset=['주소'])

splitted = pre_slate['주소'].str.split(' ', expand=True)
pre_slate['행정기관'] = splitted[2]
pre_slate['리'] = splitted[3]

pre_slate = pre_slate.dropna(subset=['행정기관'])
pre_slate = pre_slate[~pre_slate['행정기관'].str.endswith('동')]
pre_slate = pre_slate[['행정기관','리','건축물 용도','슬레이트 사용부위','철거여부']]
slate_house = pre_slate.groupby(['행정기관','리']).size().reset_index(name='슬레이트_주택수')

re_slate_house = pd.merge(count_house,slate_house,on=['행정기관','리'],how='outer')
re_slate_house['슬레이트_주택수'] = re_slate_house['슬레이트_주택수'].fillna(0.0)
re_slate_house['슬레이트주택_비율'] = ((re_slate_house['슬레이트_주택수'] / re_slate_house['전체주택수'])* 100).round(2)

result_slatehouse = re_slate_house[['행정기관','리','슬레이트주택_비율']]
#결과
result = pd.merge(result,result_slatehouse,on=['행정기관','리'])
result

### 30년이상 노후빈집,슬레이트 주택비율

In [ ]:
# 30년이상 노후빈집 비율
up30_house = pd.read_excel('../P1_Data/01.External/빈집리스트(도시, 농촌).xlsx',skiprows=1)
pre_up30_house = up30_house[['소재지','건축년도']]
splitted = pre_up30_house['소재지'].str.split(' ', expand=True)
pre_up30_house['행정기관'] = splitted[2]
pre_up30_house['리'] = splitted[3]
# "행정기관" 컬럼 값이 "동"으로 끝나는 행 삭제
pre_up30_house2 = pre_up30_house[~pre_up30_house['행정기관'].str.endswith('동')]
# 30년 이상된 노후빈집주택
up30_nullhouse = pre_up30_house2[pre_up30_house2['건축년도'] <= 1993.0 ]
cnt_up30_nullhouse = up30_nullhouse.groupby(['행정기관','리']).size().reset_index(name='노후빈집수')
UP30_NULLHUS = pd.merge(count_house,cnt_up30_nullhouse,on=['행정기관','리'],how='outer')
UP30_NULLHUS['노후빈집수'] = UP30_NULLHUS['노후빈집수'].fillna(0.0)
UP30_NULLHUS

In [ ]:
#슬레이트 주택비율
## slate(수작업).xlsx :  건축물 용도가 주택에 해당하고, 슬레이트 철거여부가 미철거에 해당하는 row만 추출
slate = pd.read_excel('../P1_Data/02.Pretreatment/slate(수작업).xlsx')
pre_slate = slate[['주소_지번','건축물대장        준공연도','건축물           용도','슬레이트            사용부위','철거           여부']]
pre_slate.columns = ['주소','건축년도','건축물 용도','슬레이트 사용부위','철거여부']
pre_slate = pre_slate.dropna(subset=['주소'])

splitted = pre_slate['주소'].str.split(' ', expand=True)
pre_slate['행정기관'] = splitted[2]
pre_slate['리'] = splitted[3]

pre_slate = pre_slate.dropna(subset=['행정기관'])
pre_slate = pre_slate[~pre_slate['행정기관'].str.endswith('동')]
pre_slate = pre_slate[['행정기관','리','건축년도','건축물 용도','슬레이트 사용부위','철거여부']]

# 30년 이상된 노후슬레이트주택
up30_slatehouse = pre_slate[pre_slate['건축년도'] <= 1993.0 ]
cnt_up30_slatehouse = up30_slatehouse.groupby(['행정기관','리']).size().reset_index(name='노후슬레이트수')


UP30_SLATEHUS = pd.merge(count_house,cnt_up30_slatehouse,on=['행정기관','리'],how='outer')
UP30_SLATEHUS['노후슬레이트수'] = UP30_SLATEHUS['노후슬레이트수'].fillna(0.0)
UP30_SLATEHUS

In [ ]:
result_30up = pd.merge(UP30_NULLHUS,UP30_SLATEHUS,on=['행정기관','리','전체주택수'])
result_30up['30년이상_빈집_슬레이트_비율'] = (((result_30up['노후빈집수'] + result_30up['노후슬레이트수']) / result_30up['전체주택수'])* 100).round(2)
result2_30up = result_30up[['행정기관','리','30년이상_빈집_슬레이트_비율']]

#결과
result = pd.merge(result,result2_30up,on=['행정기관','리'])
result

In [ ]:
#상수도 공급가구
waterworks = pd.read_excel('../P1_Data/01.External/추가데이터/5. 상수도공급가구(읍면 리별 분류).xlsx')
pre_waterworks  = waterworks.copy()
#하수도 공급가구
sewer = pd.read_excel('../P1_Data/01.External/추가데이터/하수도 사용 지역(리구분).xlsx')
pre_sewer = sewer.copy()

In [ ]:
# 조건에 해당하는 행 삭제
pre_waterworks.drop(pre_waterworks[pre_waterworks['리이름'] == '제외'].index, inplace=True)
# 조건에 해당하는 행 삭제
pre_sewer.drop(pre_sewer[pre_sewer['구분'] == '제외'].index, inplace=True)

In [ ]:
pre_waterworks_group = pre_waterworks.groupby('리이름').size().reset_index(name='상수도_가구수')
pre_waterworks_group = pre_waterworks_group.rename(columns={'리이름':'리'})
result_sangsu = pd.merge(count_house,pre_waterworks_group,on='리',how='outer')
result_sangsu['상수도_가구수'] = result_sangsu['상수도_가구수'].fillna(0.0)

In [ ]:
pre_sewer_gorup = pre_sewer.groupby('구분').size().reset_index(name='하수도_가구수')
pre_sewer_gorup = pre_sewer_gorup.rename(columns={'구분':'리'})
result_sanghasudo = pd.merge(result_sangsu,pre_sewer_gorup,on='리',how='outer')
result_sanghasudo['하수도_가구수'] = result_sanghasudo['하수도_가구수'].fillna(0.0)

In [ ]:
result_sanghasudo['상수도보급률'] = ((result_sanghasudo['상수도_가구수'] / result_sanghasudo['전체주택수'])* 100).round(2)
result_sanghasudo['하수도보급률'] = ((result_sanghasudo['하수도_가구수'] / result_sanghasudo['전체주택수'])* 100).round(2)

In [ ]:
result_sanghasudo2 = result_sanghasudo[['행정기관','리','상수도보급률','하수도보급률']]

In [ ]:
#결과
result = pd.merge(result,result_sanghasudo2,on=['행정기관','리'])
result

### 밀집도

In [ ]:
#밀집도
## 리전체주택 / 200m이내 주택수 비율 산출
## 리전체주택 / 400m이내 주택수 비율 산출
# ****** 현재 호수데이터 없음 ******

#리단위 전체 주택
total_housing = pd.read_excel('../P1_Data/01.External/추가데이터/10. 리 단위 전체 주택별 주소정보(건축물대장).xlsx',skiprows=7)
total_housing = total_housing[total_housing['주용도코드명'].isin(['단독주택', '공동주택'])] #단독주택 공동주책
total_housing['코드'] = total_housing['시군구코드'].astype(str) + total_housing['법정동코드'].astype(str)
total_housing[['행정기관', '리']] = code['법정동코드명'].str.split(' ', n=1, expand=True)

address = total_housing[['코드','행정기관','리','도로명주소']]
#지오코딩 전 주택주소 저장
#address.to_csv("../P1_Data/02.Pretreatment/house_address(주소).csv",encoding='cp949',index = False)
# '도로명주소' 컬럼의 널값 개수 확인
#address['도로명주소'].isnull().sum()

In [ ]:
#지오코딩 결과 ( 전체:21518 성공:14729 실패:6789 성공율:68.44 )
house_address_geo = pd.read_csv("../P1_Data/02.Pretreatment/house_address(지오코딩).csv",encoding='cp949')
house_address_geo = house_address_geo.dropna(subset=['c_도로명주소'])
 #좌표점이 찍힌 주택 수 : 14729행

pre_house_address_geo = house_address_geo[house_address_geo['c_시군구명'] == '공주시']
pre_house_address_geo #좌표점이 찍힌 주택 수 중 공주시 농어촌지역에 해당하는 주택 rows : 14719행

# 위도 경도 정보를 기반으로 Geopandas의 Geodataframe으로 변환
geometry = gpd.points_from_xy(pre_house_address_geo['c_경도'], pre_house_address_geo['c_위도'])
house_geometry = gpd.GeoDataFrame(pre_house_address_geo, geometry=geometry)

house_geometry = house_geometry[['코드','행정기관','리','geometry']]
house_geometry
# GeoPackage 파일로 저장
house_geometry.to_file('../P3_Result/03.SHP/house_geometry.gpkg', driver='GPKG')

In [ ]:
#리단위 행정경계면
layer_cent = gpd.read_file('../P1_Data/01.External/외부데이터/(B100)국토통계_인구정보-총 인구 수(전체)-리경계_충청남도 공주시_202304/nlsp_004001001.shp')
# 중심점 계산
layer_cent['centroid'] = layer_cent['geometry'].centroid
layer_centroid = layer_cent[['gid','lbl','val','centroid']]
layer_centroid = gpd.GeoDataFrame(layer_centroid, geometry='centroid')

# 'house_geometry' 데이터프레임의 '코드' 컬럼의 데이터 타입을 문자열로 변환
house_geometry['코드'] = house_geometry['코드'].astype(str)

# house_geometry 데이터프레임에서 코드 별 행 개수 계산하여 layer_centroid에 반영
house_counts = house_geometry.groupby('코드').size().reset_index(name='house_count')
layer_centroid = pd.merge(layer_centroid, house_counts, left_on='gid', right_on='코드', how='inner')
layer_centroid = layer_centroid[['gid', 'centroid', 'house_count']]
layer_centroid.to_file('../P3_Result/03.SHP/layer_centroid.gpkg', driver='GPKG')

In [ ]:
#200m 버퍼
buffer = gpd.GeoSeries(layer_centroid.buffer(200))
facil_buffer = pd.concat([layer_centroid, buffer], axis=1)
facil_buffer.rename(columns={0: 'buffer_geometry'}, inplace=True)
facil_buffer_geo = gpd.GeoDataFrame(facil_buffer, geometry='buffer_geometry')

facil_buffer_geo = facil_buffer_geo[['gid','house_count','centroid','buffer_geometry']] #컬럼순서 조정
facil_buffer_geo_2 = facil_buffer_geo.drop('centroid',axis=1) #200m 버퍼정보만 저장

In [ ]:
# 데이터프레임의 좌표계를 'EPSG:4326'으로 설정합니다.
house_geometry.crs = 'EPSG:4326'
# 좌표계를 'EPSG:4326'으로 변환합니다.
facil_buffer_geo_2.crs = 'EPSG:5179'
facil_buffer_geo_2 = facil_buffer_geo_2.to_crs('EPSG:4326')
facil_buffer_geo_2.to_file('../P3_Result/03.SHP/layer_200buffer.gpkg', driver='GPKG')

In [ ]:
# buffer와 houses 데이터프레임을 spatial join
#join_gdf_200 = gpd.sjoin(house_geometry, facil_buffer_geo_2, op='within', how='inner')
join_gdf_200 = gpd.sjoin(house_geometry, facil_buffer_geo_2,op='within')

In [ ]:
# 각 리 중심점으로부터 200이내의 있는 주택수 count 결과 df
in200_hus_cnt = join_gdf_200.groupby('코드').size().reset_index(name='in200_hus_cnt')
layer_centroid_2 = pd.merge(layer_centroid, in200_hus_cnt, left_on='gid', right_on='코드', how='outer')
layer_centroid_2 = layer_centroid_2[['gid','in200_hus_cnt']]
layer_centroid_2

In [ ]:
#400m 버퍼
buffer = gpd.GeoSeries(layer_centroid.buffer(400))
facil_buffer = pd.concat([layer_centroid, buffer], axis=1)
facil_buffer.rename(columns={0: 'buffer_geometry'}, inplace=True)
facil_buffer_geo = gpd.GeoDataFrame(facil_buffer, geometry='buffer_geometry')
facil_buffer_geo_4 = facil_buffer_geo.drop('centroid',axis=1) #200m 버퍼정보만 저장

In [ ]:
# 좌표계를 'EPSG:4326'으로 변환합니다.
facil_buffer_geo_4.crs = 'EPSG:5179'
facil_buffer_geo_4 = facil_buffer_geo_4.to_crs('EPSG:4326')
facil_buffer_geo_4.to_file('../P3_Result/03.SHP/layer_400buffer.gpkg', driver='GPKG')

In [ ]:
# buffer와 houses 데이터프레임을 spatial join
#join_gdf_200 = gpd.sjoin(house_geometry, facil_buffer_geo_2, op='within', how='inner')
join_gdf_400 = gpd.sjoin(house_geometry, facil_buffer_geo_4,op='within')

In [ ]:
# 각 리 중심점으로부터 400이내의 있는 주택수 count 결과 df
in400_hus_cnt = join_gdf_400.groupby('코드').size().reset_index(name='in400_hus_cnt')
layer_centroid_4 = pd.merge(layer_centroid, in400_hus_cnt, left_on='gid', right_on='코드', how='outer')
layer_centroid_4 = layer_centroid_4[['gid','in400_hus_cnt']]
layer_centroid_4

In [ ]:
# 밀집도 최종
merge1 = pd.merge(layer_centroid,layer_centroid_2,on='gid')
merge2 = pd.merge(merge1,layer_centroid_4,on='gid')
columns_to_fill_with_zero = ['in200_hus_cnt', 'in400_hus_cnt']
merge2[columns_to_fill_with_zero] = merge2[columns_to_fill_with_zero].fillna(0)
merge2[columns_to_fill_with_zero] = merge2[columns_to_fill_with_zero].astype(int)

density = merge2.copy()

In [ ]:
density['밀집도_200m'] = ((density['in200_hus_cnt']/density['house_count'])* 100).round(2)
density['밀집도_400m'] = ((density['in400_hus_cnt']/density['house_count'])* 100).round(2)

#한글 리정보 매핑
lee = pre_code.drop_duplicates(subset='코드')
house_density = pd.merge(lee,density,left_on='코드',right_on='gid')
house_density = house_density[['코드','행정기관','리','밀집도_200m','밀집도_400m']]
house_density

In [ ]:
#결과
result = pd.merge(result,house_density,on=['코드','행정기관','리'])
result

In [ ]:
# 코드정렬 후 결과 저장
result=result.sort_values(by=['코드'], ascending=True)
result = result.reset_index(drop='Ture')
#엑셀 저장
result.to_csv("../P3_Result/01.CSV/공주시_산출결과.csv", encoding='cp949', index=False)
result

In [ ]:
#결과파일 복사
result_copy = result.copy()
result_copy

### 각 지표별 최솟값 최대값을 기준으로 (10개항목으로 나눠서 빈도수 세기기 )

In [ ]:
# 시각화를 위한 폰트 설정
font_list = fm.findSystemFonts(fontpaths=None, fontext='ttf')
font_list[:10]  # 일부만 출력
font_path='C:\\Windows\\Fonts'
mpl.rcParams['font.family'] = 'Malgun Gothic'

def plot_frequency(df, column_name, save_path=None):
    # 최솟값과 최댓값 계산
    min_value = 0
    max_value = df[column_name].max()

    # 네추럴브레이크로 10개의 구분값 계산 (소수점 둘째 자리로 설정)
    breaks = pd.np.linspace(min_value, max_value, num=11, endpoint=True)

    # 새로운 비어있는 데이터프레임 생성
    new_df = pd.DataFrame()

    # '구분' 컬럼 추가 (소수점 아래 1자리까지)
    # '구분' 컬럼 추가 (물결 기호로 표현되도록 labels 설정)
    labels = [f'{current:.2f} ~ {next_val:.2f}'.replace(",", " ~ ") for current, next_val in zip(breaks, breaks[1:])]
    new_df['구분'] = pd.cut(df[column_name], bins=breaks, labels=labels, include_lowest=True, precision=2)

    # 각 구분값의 빈도수 계산
    frequency = new_df['구분'].value_counts().sort_index()

    # 결과를 데이터프레임으로 변환
    result_df = pd.DataFrame({'빈도수': frequency})

    # 막대그래프 그리기
    plt.figure(figsize=(13,10))
    plt.bar(frequency.index.astype(str), frequency.values)
    plt.xlabel('빈도구분값',fontsize=13)
    plt.ylabel('빈도수',fontsize=13)
    plt.title(f'{column_name} 빈도수 막대그래프',fontsize=16)
    plt.xticks(rotation=25,fontsize=12)

    # 평균값, 최소값, 최댓값을 그래프 상에 표시
    mean_value = df[column_name].mean()
    plt.axhline(label=f'평균값: {mean_value:.2f}')
    plt.axhline(label=f'최소값: {min_value:.2f}')
    plt.axhline(label=f'최댓값: {max_value:.2f}')

    plt.legend(fontsize=14)  # 범례 추가


    # 이미지로 저장
    if save_path:
        plt.savefig(save_path)
    else:
        plt.show()

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/기초생활수급자_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '기초생활수급자_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/차상위계층_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '차상위계층_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/장애인비율_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '장애인_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/65세이상고령자비율_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '65세이상고령자_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/빈집비율_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '빈집_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/슬레이트비율_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '슬레이트주택_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/30년이상 노후주택비율.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '30년이상_빈집_슬레이트_비율', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/밀집도_200m_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '밀집도_200m', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/밀집도_400m_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '밀집도_400m', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/상수도보급률.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '상수도보급률', save_path=output_path)

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/하수도보급률.png'  # 저장할 이미지 파일 경로
plot_frequency(result_copy, '하수도보급률', save_path=output_path)

### 점수환산에 사용될 표준화 값 구하기

In [ ]:
# Min-Max 스케일링을 위한 scaler 객체 생성
scaler = MinMaxScaler()

# 모든 컬럼에 대해 정규화 수행
result_copy[result_copy.columns[3:]] = scaler.fit_transform(result_copy[result_copy.columns[3:]])
# 소수점 둘째 자리까지 반올림
result_minmax = result_copy.round(2)


result_minmax=result_minmax.sort_values(by=['코드'], ascending=True)
#표준화 결과 엑셀 저장
result_minmax.to_csv("../P3_Result/01.CSV/공주시_산출결과_표준화.csv",encoding='cp949',index = False)

In [ ]:
result_minmax

### 점수환산

In [ ]:
# 빈 데이터프레임 생성 (컬럼 이름만 지정)
result_score = result_minmax.copy()
# 각 항목별 비율 구하기
result_score['취약계층비율_기초생활수급자/차상위'] = result_score[['기초생활수급자_비율', '차상위계층_비율']].mean(axis = 1).round(2)
result_score['취약계층비율_고령자/장애인'] = result_score[['장애인_비율','65세이상고령자_비율']].mean(axis = 1).round(2)
result_score['생활위생인프라_상수도'] = result_score[['상수도보급률']].mean(axis = 1).round(2)
result_score['생활위생인프라_하수도'] = result_score[['하수도보급률']].mean(axis = 1).round(2)
result_score['주택취약도_노후주택(빈집,슬레이트)'] = result_score[['30년이상_빈집_슬레이트_비율']].mean(axis = 1).round(2)
result_score['주택취약도_슬레이트'] = result_score[['슬레이트주택_비율']].mean(axis = 1).round(2)
result_score['주택취약도_공폐가'] = result_score[['빈집_비율']].mean(axis = 1).round(2)
result_score['밀집도_200m기준'] = result_score[['밀집도_200m']].mean(axis = 1).round(2)


result_score2 = result_score[['코드','행정기관','리','취약계층비율_기초생활수급자/차상위','취약계층비율_고령자/장애인','생활위생인프라_상수도','생활위생인프라_하수도','주택취약도_노후주택(빈집,슬레이트)','주택취약도_슬레이트','주택취약도_공폐가','밀집도_200m기준']]

# 각 컬럼에 값 곱하기
result_score2['취약계층비율_기초생활수급자/차상위'] *= 6
result_score2['취약계층비율_고령자/장애인'] *= 4
result_score2['생활위생인프라_상수도'] = (1 - result_score2['생활위생인프라_상수도']) * 4
result_score2['생활위생인프라_하수도'] = (1 - result_score2['생활위생인프라_하수도']) * 4
result_score2['주택취약도_노후주택(빈집,슬레이트)'] *= 5
result_score2['주택취약도_슬레이트'] *= 5
result_score2['주택취약도_공폐가'] *= 2
result_score2['밀집도_200m기준'] *= 4

# 총합점수 계산하여 새로운 컬럼에 추가
result_score2['총합점수'] = result_score2[['취약계층비율_기초생활수급자/차상위', '취약계층비율_고령자/장애인','생활위생인프라_상수도','생활위생인프라_하수도','주택취약도_노후주택(빈집,슬레이트)','주택취약도_슬레이트', '주택취약도_공폐가','밀집도_200m기준']].sum(axis=1)
#엑셀 저장
result_score2.to_csv("../P3_Result/01.CSV/공주시_항목별_점수.csv",encoding='cp949',index = False)
result_score2

In [ ]:
# 함수 호출하여 막대그래프 그리고 이미지로 저장
output_path = '../P3_Result/02.PNG/총합점수_빈도수.png'  # 저장할 이미지 파일 경로
plot_frequency(result_score2, '총합점수', save_path=output_path)

In [ ]:
result

In [ ]:
#리단위 행정경계면 저장
layer = gpd.read_file('../P1_Data/01.External/외부데이터/(B100)국토통계_인구정보-총 인구 수(전체)-리경계_충청남도 공주시_202304/nlsp_004001001.shp',encoding='cp949')
#산출값 결과에 경계데이터 매핑
result_geo = pd.merge(result,layer,left_on= '코드',right_on='gid')
result_geo = gpd.GeoDataFrame(result_geo, geometry='geometry')
result_geo = result_geo[['코드', '행정기관', '리', '기초생활수급자_비율', '차상위계층_비율', '장애인_비율','상수도보급률','하수도보급률', '65세이상고령자_비율','빈집_비율', '슬레이트주택_비율','30년이상_빈집_슬레이트_비율' ,'밀집도_200m', '밀집도_400m','geometry']]
result_geo.to_file('../P3_Result/03.SHP/result_geo.gpkg', driver='GPKG')
result_geo

In [ ]:
#점수산출 결과에 경계데이터 매핑
#산출값 결과에 경계데이터 매핑
result_score_geo = pd.merge(result_score2,layer,left_on= '코드',right_on='gid')
result_score_geo = gpd.GeoDataFrame(result_score_geo, geometry='geometry')
result_score_geo = result_score_geo[['코드', '행정기관', '리','취약계층비율_기초생활수급자/차상위', '취약계층비율_고령자/장애인','생활위생인프라_상수도','생활위생인프라_하수도','주택취약도_노후주택(빈집,슬레이트)','주택취약도_슬레이트', '주택취약도_공폐가','밀집도_200m기준','총합점수','geometry']]
result_score_geo.to_file('../P3_Result/03.SHP/result_score_geo.gpkg', driver='GPKG')
result_score_geo